# 실습 1. Voting 앙상블

## 데이터
- 분류: `sklearn.datasets.load_breast_cancer()`
  - 종양의 반지름, 질감, 면적, 오목도 등 30개 수치 feature로 악성/양성을 분류함.
  - target
    - `0`: malignant, 악성
    - `1`: benign, 양성
- 회귀: `sklearn.datasets.load_diabetes()`
  - 당뇨병 진행 정도를 예측하는 회귀 데이터셋임.

## 실습 목표
- Voting 앙상블에 사용할 학습/평가 데이터와 스케일링 데이터를 준비함.
- 서로 다른 개별 분류 모델의 성능을 비교함.
- Hard Voting이 class 다수결로 예측하는 방식을 확인함.
- Soft Voting이 class별 확률 평균으로 예측하는 방식을 확인함.
- VotingRegressor가 여러 회귀 모델의 숫자 예측값을 평균내는 방식을 확인함.


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingClassifier, VotingRegressor
from sklearn.metrics import classification_report, root_mean_squared_error, r2_score

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

print('분류 X:', X.shape)
print('분류 y:', y.shape)
print('target names:', cancer.target_names)
display(X.head())


분류 X: (569, 30)
분류 y: (569,)
target names: ['malignant' 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 문제 1. 학습/평가 데이터 분리와 스케일링

Breast Cancer 분류 데이터를 학습용과 평가용으로 나누고, feature 스케일을 맞추세요.

### 요구사항
- `train_test_split()` 사용
- `test_size=0.2` 사용
- `random_state=42` 사용
- `stratify=y` 사용
- `StandardScaler()` 사용
- 훈련 데이터에는 `fit_transform()`, 평가 데이터에는 `transform()` 적용
- 분리 결과 shape 출력

### 힌트
- Logistic Regression과 KNN은 feature 스케일 영향을 받을 수 있음.
- 평가 데이터에 `fit_transform()`을 사용하면 평가 데이터 정보가 전처리 기준에 섞일 수 있음.

### 실행 결과
```text
X_train_scaled: (455, 30)
X_test_scaled: (114, 30)
y_train: (455,)
y_test: (114,)
```


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('X_train_scaled:', X_train_scaled.shape)
print('X_test_scaled:', X_test_scaled.shape)
print('y_train', y_train.shape)
print('y_test', y_test.shape)

X_train_scaled: (455, 30)
X_test_scaled: (114, 30)
y_train (455,)
y_test (114,)


## 문제 2. 개별 분류 모델 성능 비교

Voting에 넣을 개별 모델을 만들고, 각 모델의 학습셋/평가셋 accuracy를 비교하세요.

### 요구사항
- Logistic Regression: `LogisticRegression(max_iter=5000, random_state=42)`
- KNN: `KNeighborsClassifier(n_neighbors=5)`
- Decision Tree: `DecisionTreeClassifier(max_depth=4, random_state=42)`
- 결과 컬럼: `model`, `train_accuracy`, `test_accuracy`
- 결과를 DataFrame으로 출력

### 해석 포인트
- Voting은 개별 모델의 예측을 결합하므로, 먼저 개별 모델의 성능을 확인해야 함.
- 서로 다른 방식의 모델을 섞어야 예측 관점이 다양해질 수 있음.

### 실행 결과
```text
model                train_accuracy  test_accuracy
logistic_regression  0.989011        0.982456
knn                  0.973626        0.956140
decision_tree        0.986813        0.938596
```


In [4]:
lr_clf = LogisticRegression(max_iter=5000, random_state=42)
knn_clf = KNeighborsClassifier(n_neighbors=5)
dt_clf = DecisionTreeClassifier(max_depth=4, random_state=42)

base_classifiers = {
    'logistic_regression': lr_clf,
    'knn': knn_clf,
    'decision_tree': dt_clf
}

results = []
for name, model in base_classifiers.items():
    model.fit(X_train_scaled, y_train)
    results.append({
        'model': name,
        'train_accuracy': model.score(X_train_scaled, y_train),
        'test_accuracy': model.score(X_test_scaled, y_test)
    })

results_df = pd.DataFrame(results)
display(results_df)

,model,train_accuracy,test_accuracy
0,logistic_regression,0.989011,0.982456
1,knn,0.973626,0.956140
2,decision_tree,0.986813,0.938596


## 문제 3. Hard Voting 모델 학습과 다수결 결과 확인

`VotingClassifier(voting='hard')`를 사용해 Hard Voting 모델을 학습하고, 일부 샘플의 예측 결과를 비교하세요.

### 요구사항
- estimators에 Logistic Regression, KNN, Decision Tree 사용
- `voting='hard'` 사용
- Hard Voting의 학습셋/평가셋 accuracy 출력
- 평가 데이터 일부 샘플에 대해 실제값, 개별 모델 예측, Hard Voting 예측을 표로 출력
- `classification_report()` 출력

### 힌트
- Hard Voting은 가장 성능 좋은 모델을 고르는 방식이 아님.
- 샘플마다 개별 모델의 class 예측을 다수결로 합쳐 최종 예측을 만듦.

### 실행 결과
```text
Hard Voting 학습셋 accuracy: 0.9868131868131869
Hard Voting 평가셋 accuracy: 0.9824561403508771

actual  lr_pred  knn_pred  dt_pred  hard_voting_pred
0       0        0         0        0
0       0        0         0        0
1       1        1         1        1
1       1        1         1        1
1       1        1         1        1
```


In [22]:
hard_voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_clf),
        ('knn', knn_clf),
        ('dt', dt_clf),
    ],
    voting='hard',
)
# 학습
hard_voting_clf.fit(X_train_scaled, y_train)

# 평가
print('Hard Voting 학습셋 accuracy', hard_voting_clf.score(X_train_scaled, y_train))
print('Hard Voting 평가셋 accuracy', hard_voting_clf.score(X_test_scaled, y_test))


sample_start = 20
sample_stop = 25
hard_pred_df = pd.DataFrame({
    'actual': y_test.to_numpy()[sample_start:sample_stop],
    'lr_pred': lr_clf.predict(X_test_scaled[sample_start:sample_stop]),
    'knn_pred': knn_clf.predict(X_test_scaled[sample_start:sample_stop]),
    'dt_pred': dt_clf.predict(X_test_scaled[sample_start:sample_stop]),
    'hard_voting_pred': hard_voting_clf.predict(X_test_scaled[sample_start:sample_stop])
})
display(hard_pred_df)

hard_y_pred = hard_voting_clf.predict(X_test_scaled)

print(classification_report(y_test, hard_y_pred, target_names=cancer.target_names))


Hard Voting 학습셋 accuracy 0.9868131868131869
Hard Voting 평가셋 accuracy 0.9824561403508771


,actual,lr_pred,knn_pred,dt_pred,hard_voting_pred
0,0,0,0,0,0
1,0,0,0,0,0
2,1,1,1,1,1
3,1,1,1,1,1
4,1,1,1,1,1


              precision    recall  f1-score   support

   malignant       1.00      0.95      0.98        42
      benign       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## 문제 4. Soft Voting 확률 평균 확인

`VotingClassifier(voting='soft')`를 사용해 Soft Voting 모델을 학습하고, class 확률 평균이 어떻게 계산되는지 확인하세요.

### 요구사항
- estimators에 Logistic Regression, KNN, Decision Tree 사용
- `voting='soft'` 사용
- Soft Voting의 학습셋/평가셋 accuracy 출력
- 학습된 내부 모델의 `predict_proba()` 결과를 사용해 class 1 확률 확인
- 개별 모델 확률 평균과 `soft_voting_clf.predict_proba()` 결과 비교

### 해석 포인트
- Soft Voting은 class 예측값만 보지 않고, 각 class에 속할 확률까지 평균냄.
- `manual_avg_proba`와 `soft_voting_proba`가 같으면 Soft Voting 계산 흐름을 확인한 것임.

### 실행 결과
```text
Soft Voting 학습셋 accuracy: 0.989010989010989
Soft Voting 평가셋 accuracy: 0.956140350877193

actual  lr_proba_class_1  knn_proba_class_1  dt_proba_class_1  manual_avg_proba  soft_voting_proba  soft_voting_pred
0       0.002             0.0                0.000             0.001             0.001              0
0       0.052             0.0                0.000             0.017             0.017              0
1       1.000             1.0                0.992             0.997             0.997              1
1       0.999             1.0                0.992             0.997             0.997              1
1       0.990             1.0                0.992             0.994             0.994              1
```


In [24]:
soft_voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=5000, random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('dt', DecisionTreeClassifier(max_depth=4, random_state=42)),
    ],
    voting='soft'
)
# 학습
soft_voting_clf.fit(X_train_scaled, y_train)


print('Soft Voting 학습셋 accuracy:', soft_voting_clf.score(X_train_scaled, y_train))
print('Soft Voting 평가셋 accuracy:', soft_voting_clf.score(X_test_scaled, y_test))

trained_lr, trained_knn, trained_dt = soft_voting_clf.estimators_

lr_proba = trained_lr.predict_proba(X_test_scaled[sample_start:sample_stop])[:, 1]
knn_proba = trained_knn.predict_proba(X_test_scaled[sample_start:sample_stop])[:, 1]
dt_proba = trained_dt.predict_proba(X_test_scaled[sample_start:sample_stop])[:, 1]
soft_proba = soft_voting_clf.predict_proba(X_test_scaled[sample_start:sample_stop])[:, 1]

soft_proba_df = pd.DataFrame({
    'actual': y_test.to_numpy()[sample_start:sample_stop],
    'lr_proba_class_1': lr_proba,
    'knn_proba_class_1': knn_proba,
    'dt_proba_class_1': dt_proba,
    'manual_avg_proba': (lr_proba + knn_proba + dt_proba) / 3,
    'soft_voting_proba': soft_proba,
    'soft_voting_pred': soft_voting_clf.predict(X_test_scaled[sample_start:sample_stop])
})

display(soft_proba_df.round(3))

Soft Voting 학습셋 accuracy: 0.989010989010989
Soft Voting 평가셋 accuracy: 0.956140350877193


,actual,lr_proba_class_1,knn_proba_class_1,dt_proba_class_1,manual_avg_proba,soft_voting_proba,soft_voting_pred
0,0,0.002,0.0,0.000,0.001,0.001,0
1,0,0.052,0.0,0.000,0.017,0.017,0
2,1,1.000,1.0,0.992,0.997,0.997,1
3,1,0.999,1.0,0.992,0.997,0.997,1
4,1,0.990,1.0,0.992,0.994,0.994,1


## 문제 5. VotingRegressor로 회귀 모델 결합

Diabetes 회귀 데이터에서 여러 회귀 모델의 예측값을 평균내는 VotingRegressor를 학습하세요.

### 요구사항
- `load_diabetes(as_frame=True)` 사용
- `train_test_split(test_size=0.2, random_state=42)` 사용
- `StandardScaler()`로 feature 스케일링
- 개별 모델
  - `LinearRegression()`
  - `KNeighborsRegressor(n_neighbors=5)`
  - `DecisionTreeRegressor(max_depth=4, random_state=42)`
- `VotingRegressor`로 세 모델 결합
- 모델별 RMSE와 R2를 표로 비교
- 평가 데이터 앞 3개 샘플의 실제값과 VotingRegressor 예측값 출력

### 실행 결과
```text
model                    RMSE       R2
voting_regressor         52.526867  0.479239
linear_regression        53.853446  0.452603
knn_regressor            55.203713  0.424809
decision_tree_regressor  59.740817  0.326375

실제값: 219.0, 예측값: 146.83
실제값: 70.0, 예측값: 174.16
실제값: 202.0, 예측값: 154.12
```


In [29]:
diabetes = load_diabetes(as_frame=True)

reg_X = diabetes.data
reg_y = diabetes.target

reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X,
    reg_y,
    test_size=0.2,
    random_state=42
)

# 스케일링
reg_scaler = StandardScaler()
reg_X_train_scaled = reg_scaler.fit_transform(reg_X_train)
reg_X_test_scaled = reg_scaler.transform(reg_X_test)

# 일반 회귀 모델 생성
lin_reg = LinearRegression()
knn_reg = KNeighborsRegressor(n_neighbors=5)
dt_reg = DecisionTreeRegressor(max_depth=4, random_state=42)

# VotingRegressor는 voting 방식 지정 X
# -> 입력값(X)에 대한 각 모델의 예측값(y_pred)의 평균을 구한다
voting_reg = VotingRegressor(
    estimators=[
        ('linear', lin_reg),
        ('knn', knn_reg),
        ('dt', dt_reg),
    ]
)

reg_models = {
    'voting_regressor': voting_reg,
    'linear_regression': lin_reg,
    'knn_regressor': knn_reg,
    'decision_tree_regressor': dt_reg
}

reg_results = []

# 모델들을 순서대로 학습시키고 RMSE, R2 값 구하기
for name, model in reg_models.items():
    model.fit(reg_X_train_scaled, reg_y_train)

    pred = model.predict(reg_X_test_scaled)

    reg_results.append({
        'model': name,
        'RMSE': root_mean_squared_error(reg_y_test, pred),
        'R2': r2_score(reg_y_test, pred)
    })

reg_result_df = pd.DataFrame(reg_results).sort_values('RMSE')
display(reg_result_df)

for actual, pred in zip(reg_y_test[:3], voting_reg.predict(reg_X_test_scaled[:3])):
    print(f'실제값: {actual}, 예측값: {pred:.2f}')

,model,RMSE,R2
0,voting_regressor,52.526867,0.479239
1,linear_regression,53.853446,0.452603
2,knn_regressor,55.203713,0.424809
3,decision_tree_regressor,59.740817,0.326375


실제값: 219.0, 예측값: 146.83
실제값: 70.0, 예측값: 174.16
실제값: 202.0, 예측값: 154.12
